# Projected Totals Statistics Comparison

This notebook loads the latest matching optimized simulation pickles for the campaign, builds a run-level KPI table from `projected_totals_stats` (supplemented with selected physical KPIs), and produces overview and strategy-comparison figures.

## Notebook workflow

Run the notebook **top to bottom** after adjusting `PIPELINE_CONFIG` if needed:

1. **Setup and campaign configuration** — imports, KPI definitions, data-selection settings, and reusable pipeline helpers.
2. **Data discovery and loading** — load selected pickles, construct the comparison tables, and inspect selection diagnostics.
3. **Data-quality and table review** — inspect extracted KPIs, scoring provenance, and optional CSV outputs.
4. **Run-level overview figures** — bar charts, run-level heatmap, and radar charts.
5. **Mode-level aggregated figures** — KPI ranges/lollipop overview and mean-KPI heatmap.
6. **Strategy comparison figures** — strategy heatmaps and deltas relative to constant feed/perfusion baselines.

> Optional exports are controlled locally by the existing `EXPORT_*`, `SAVEFIG`, and `SAVE_FIG` flags. Their output paths and filenames are intentionally retained.


## 1. Setup and Campaign Configuration

### 1.1 Imports and campaign paths

In [1]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import re
from datetime import datetime

# Resolve the campaign root from this notebook's location
NOTEBOOK_DIR  = Path().resolve()          # 06_Analysis/
CAMPAIGN_DIR  = NOTEBOOK_DIR.parent       # Campaign_1_mode_comparison_fixedstoichiometry/

print(f"Campaign root : {CAMPAIGN_DIR}")
print(f"Notebook dir  : {NOTEBOOK_DIR}")

Campaign root : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_2_fixed_stoichiometry_GFI
Notebook dir  : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_2_fixed_stoichiometry_GFI\06_Analysis


In [2]:
# Derive campaign identity from the notebook location so this notebook can be copied as-is
CAMPAIGN_FOLDER_NAME = CAMPAIGN_DIR.name
m = re.search(r"campaign[_\-]?(\d+)", CAMPAIGN_FOLDER_NAME.lower())
CAMPAIGN_LABEL = f"Campaign_{m.group(1)}" if m else CAMPAIGN_FOLDER_NAME

# Main canonical DCM KPIs. Conversion costs are lower-is-better.
MAIN_KPIS = [
    "STY_kg_m3_day",
    "total_DCM_harvested_kg",
    "medium_conversion_efficiency_m3_kg",
    "feed_conversion_efficiency_kg_kg",
    "protein_conversion_efficiency_kg_kg",
    "number_of_batches",
]

SUMMARY_STAT_EXPORT_ORDER = [
    "STY_kg_m3_day",
    "total_DCM_harvested_kg",
    "medium_conversion_efficiency_m3_kg",
    "feed_conversion_efficiency_kg_kg",
    "protein_conversion_efficiency_kg_kg",
    "total_simulation_time_h",
    "total_DCM_lost_kg",
    "total_base_in_L",
    "total_acid_in_L",
    "total_medium_in_L",
    "number_of_batches",
    "projection_mode",
    "projection_scale_factor",
    "projection_window_h",
    "projection_target_time_h",
    "last_batch_termination_reason",
    "unique_batch_termination_reasons",
    "batch_time_min_h",
    "batch_time_max_h",
    "batch_time_range_h",
    "last_batch_final_volume_L",
    "last_batch_final_volume_fraction",
]

SUMMARY_META_COLS = [
    "simulation_type", "timestamp", "mode", "run_id", "sweep", "run_key",
    "export_folder", "export_timestamp",
]

KPI_LABELS_MAIN = {
    "STY_kg_m3_day": "STY (kg/m3/day)",
    "total_DCM_harvested_kg": "Harvested Dry Cell Mass (kg)",
    "medium_conversion_efficiency_m3_kg": "Medium Conversion (m3/kg DCM)",
    "feed_conversion_efficiency_kg_kg": "Feed Conversion (kg feed/kg DCM)",
    "protein_conversion_efficiency_kg_kg": "Protein Conversion (kg Gln+AA/kg protein)",
    "total_simulation_time_h": "Total Simulation Time (h)",
    "number_of_batches": "Number of Batches",
}

MODE_ORDER = ["FedBatch", "Batch", "Chemostat", "Concentrated FedBatch", "Continuous Perfusion"]

PIPELINE_CONFIG = {
    "campaign_dir": CAMPAIGN_DIR,
    "campaign_label": CAMPAIGN_LABEL,
    "exclude_patterns": [],
    "keep_latest_only": True,
    "skip_sensitivity_pickles": True,
    "skip_notebook_folder": "06_Analysis",
    "sweep_filter": "p2",
    "latest_scope": "per_run",
}

print("Using projected_totals_stats as KPI source.")
print(f"Configured sweep filter: {PIPELINE_CONFIG['sweep_filter']!r}")

Using projected_totals_stats as KPI source.
Configured sweep filter: 'p2'


In [3]:
# Helper functions for the campaign pipeline
EXPORT_FOLDER_RE = re.compile(
    r"^(?P<run_key>.+?)_(?P<sweep_token>[^_]+)_(?P<date>\d{8})_(?P<time>\d{4})$",
    re.IGNORECASE,
)
SWEEP_TOKEN_RE = re.compile(r"(?<![A-Za-z0-9])(?P<token>[A-Za-z]+\d+|opt)(?![A-Za-z0-9])", re.IGNORECASE)
SWEEP_ALIASES = {"opt": "undefined"}


def canonical_mode(mode_raw: str) -> str:
    s = str(mode_raw).lower()
    s = re.sub(r"^\d+[_\-\s]*", "", s)
    s = s.replace("_", "").replace("-", "").replace(" ", "")
    if "concentratedfedbatch" in s or s.startswith("cf"):
        return "Concentrated FedBatch"
    if "continuousperfusion" in s or s.startswith("cp"):
        return "Continuous Perfusion"
    if "fedbatch" in s:
        return "FedBatch"
    if "chemostat" in s:
        return "Chemostat"
    if "batch" in s:
        return "Batch"
    return str(mode_raw)


def normalize_sweep_token(token) -> str:
    token = str(token).strip().lower()
    return SWEEP_ALIASES.get(token, token)


def detect_sweep_tokens(path: Path, campaign_dir: Path) -> list[tuple[str, str]]:
    """Return raw and normalized sweep tokens from the path, preferring folders.

    Exported pickle files can have a generic ``opt_*.pkl`` name while the parent
    folder carries the actual sweep, for example ``fedbatch_constant_p2_*``.
    """
    relative = str(path.relative_to(campaign_dir)).replace("\\", "/")
    directory_parts = relative.split("/")[:-1]
    file_stem = Path(relative.split("/")[-1]).stem
    matches: list[tuple[str, str]] = []

    for component in directory_parts:
        for match in SWEEP_TOKEN_RE.finditer(component):
            raw = match.group("token").lower()
            matches.append((raw, normalize_sweep_token(raw)))

    if matches:
        return matches

    for match in SWEEP_TOKEN_RE.finditer(file_stem):
        raw = match.group("token").lower()
        matches.append((raw, normalize_sweep_token(raw)))
    return matches


def parse_pickle_metadata(pkl_path: Path, campaign_dir: Path) -> dict:
    """Parse run identity and detect sweep tokens anywhere in the relative path."""
    export_folder = pkl_path.parent.name
    match = EXPORT_FOLDER_RE.match(export_folder)
    rel = pkl_path.relative_to(campaign_dir)
    mode_folder = rel.parts[0] if rel.parts else "."
    detected = detect_sweep_tokens(pkl_path, campaign_dir)

    if match:
        run_key = match.group("run_key")
        export_timestamp = f"{match.group('date')}_{match.group('time')}"
    else:
        run_key = pkl_path.stem
        export_timestamp = ""

    if detected:
        raw_sweep, sweep = detected[-1]
    else:
        raw_sweep, sweep = "unknown", "unknown"

    return {
        "path": pkl_path,
        "mode_folder": mode_folder,
        "mode": canonical_mode(mode_folder),
        "run_key": run_key,
        "sweep": sweep,
        "raw_sweep_token": raw_sweep,
        "detected_sweep_tokens": " | ".join(raw for raw, _ in detected),
        "export_folder": export_folder,
        "export_timestamp": export_timestamp,
        "file_stem": pkl_path.stem,
        "relative_path": str(rel).replace("\\", "/"),
    }


def make_label(pkl_path: Path, campaign_dir: Path) -> str:
    metadata = parse_pickle_metadata(pkl_path, campaign_dir)
    return f"{metadata['mode_folder']} / {metadata['run_key']}_{metadata['raw_sweep_token']}"


def sweep_matches(metadata: dict, sweep_filter) -> bool:
    if sweep_filter is None or str(sweep_filter).strip().lower() == "all":
        return True
    requested = sweep_filter if isinstance(sweep_filter, (list, tuple, set)) else [sweep_filter]
    requested = {normalize_sweep_token(value) for value in requested if str(value).strip()}
    return metadata["sweep"] in requested


def is_excluded(pkl_path: Path, campaign_dir: Path, exclude: list[str]) -> bool:
    rel_str = str(pkl_path.relative_to(campaign_dir)).replace("\\", "/").lower()
    return any(ex.lower() in rel_str for ex in exclude)


def filter_latest(pkl_files: list[Path], campaign_dir: Path) -> list[Path]:
    """Keep the newest pickle per mode, run key, and normalized sweep."""
    latest: dict[tuple[str, str, str], Path] = {}
    for p in pkl_files:
        metadata = parse_pickle_metadata(p, campaign_dir)
        key = (metadata["mode_folder"], metadata["run_key"], metadata["sweep"])
        if key not in latest or p.stat().st_mtime > latest[key].stat().st_mtime:
            latest[key] = p
    return sorted(latest.values())


def normalize_summary_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, pd.Timestamp):
        value = value.isoformat()
    if isinstance(value, datetime):
        value = value.isoformat()
    if isinstance(value, set):
        return " | ".join(str(item) for item in sorted(value, key=str))
    if isinstance(value, (list, tuple)):
        return " | ".join(str(item) for item in value)
    if isinstance(value, Path):
        return str(value)
    return value


def extract_summary_stats(obj, label: str) -> dict:
    if not hasattr(obj, "projected_totals_stats") or not isinstance(obj.projected_totals_stats, dict) or not obj.projected_totals_stats:
        return {}
    row = dict(obj.projected_totals_stats)
    if hasattr(obj, "summary_stats") and isinstance(obj.summary_stats, dict):
        for k in (
            "STY_kg_m3_day",
            "medium_use_efficiency_kg_m3",
            "nutrient_use_efficiency_kg_kg",
            "protein_use_efficiency_kg_kg",
        ):
            if k in obj.summary_stats:
                row[k] = obj.summary_stats[k]
    if hasattr(obj, "simulation_type"):
        row["simulation_type"] = obj.simulation_type
    if hasattr(obj, "timestamp"):
        row["timestamp"] = obj.timestamp
    return {key: normalize_summary_value(value) for key, value in row.items()}


def build_campaign_summary_table(df_stats: pd.DataFrame, campaign_label: str) -> pd.DataFrame:
    if "mode" not in df_stats.columns:
        raise ValueError("df_stats must contain a 'mode' column.")
    available_kpis = [k for k in MAIN_KPIS if k in df_stats.columns]
    if not available_kpis:
        raise ValueError("None of MAIN_KPIS are available in df_stats.")
    work = df_stats.copy()
    work["mode_group"] = work["mode"].map(canonical_mode)
    long_df = work.melt(
        id_vars=["mode_group"],
        value_vars=available_kpis,
        var_name="kpi",
        value_name="value",
    )
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.dropna(subset=["value"])
    agg = (
        long_df.groupby(["mode_group", "kpi"], as_index=False)["value"]
        .agg(min_value="min", max_value="max", mean_value="mean", n_runs="count")
    )
    agg.insert(0, "campaign", campaign_label)
    agg["kpi_label"] = agg["kpi"].map(KPI_LABELS_MAIN).fillna(agg["kpi"])
    agg["mode_group"] = pd.Categorical(agg["mode_group"], categories=MODE_ORDER, ordered=True)
    return agg.sort_values(["kpi", "mode_group"]).reset_index(drop=True)


def run_campaign_pipeline(config: dict) -> dict:
    campaign_dir = Path(config["campaign_dir"])
    exclude = config.get("exclude_patterns", [])
    keep_latest_only = bool(config.get("keep_latest_only", True))
    skip_sensitivity = bool(config.get("skip_sensitivity_pickles", True))
    skip_folder = str(config.get("skip_notebook_folder", "06_Analysis"))
    sweep_filter = config.get("sweep_filter", "all")
    if sweep_filter is not None and not isinstance(sweep_filter, (list, tuple, set)) and not str(sweep_filter).strip():
        raise ValueError("sweep_filter must be a non-empty value, 'all', or None.")

    all_pkl_files = sorted(
        p for p in campaign_dir.rglob("*.pkl")
        if skip_folder not in p.parts and (not skip_sensitivity or "_sens_" not in p.stem.lower())
    )
    candidate_files = [p for p in all_pkl_files if not is_excluded(p, campaign_dir, exclude)]
    metadata_by_path = {p: parse_pickle_metadata(p, campaign_dir) for p in candidate_files}
    sweep_candidate_files = [p for p in candidate_files if sweep_matches(metadata_by_path[p], sweep_filter)]
    selected_files = filter_latest(sweep_candidate_files, campaign_dir) if keep_latest_only else sweep_candidate_files

    available_sweeps = sorted({m["sweep"] for m in metadata_by_path.values()})
    requested = sweep_filter if isinstance(sweep_filter, (list, tuple, set)) else [sweep_filter]
    requested = {normalize_sweep_token(value) for value in requested if value is not None and str(value).lower() != "all"}
    if requested and not requested.intersection(available_sweeps):
        print(f"Warning: sweep filter {sorted(requested)} matched no detected sweeps. Available: {available_sweeps}")

    loaded: dict[str, object] = {}
    errors: dict[str, str] = {}
    for pkl_path in selected_files:
        label = make_label(pkl_path, campaign_dir)
        try:
            with open(pkl_path, "rb") as fh:
                loaded[label] = pickle.load(fh)
        except Exception as exc:
            errors[label] = str(exc)

    records: dict[str, dict] = {}
    for pkl_path in selected_files:
        label = make_label(pkl_path, campaign_dir)
        if label not in loaded:
            continue
        data_obj = loaded[label].data if hasattr(loaded[label], "data") else loaded[label]
        stats = extract_summary_stats(data_obj, label)
        if stats:
            metadata = metadata_by_path[pkl_path]
            stats.update({
                key: value for key, value in metadata.items()
                if key not in {"path", "file_stem", "relative_path", "raw_sweep_token", "mode_folder"}
            })
            stats["mode"] = metadata["mode"]
            stats["run_id"] = metadata["run_key"]
            records[label] = stats

    df_all = pd.DataFrame.from_dict(records, orient="index")
    if not df_all.empty:
        df_all.index.name = "run"
        ordered_cols = [c for c in SUMMARY_META_COLS + SUMMARY_STAT_EXPORT_ORDER if c in df_all.columns]
        extra_cols = sorted(c for c in df_all.columns if c not in ordered_cols)
        df_stats = df_all[ordered_cols + extra_cols].copy()
    else:
        df_stats = pd.DataFrame()

    return {
        "all_pkl_files": all_pkl_files,
        "candidate_files": candidate_files,
        "sweep_candidate_files": sweep_candidate_files,
        "selected_files": selected_files,
        "metadata_by_path": metadata_by_path,
        "loaded": loaded,
        "errors": errors,
        "records": records,
        "df_all": df_all,
        "df_stats": df_stats,
    }

print(f"Pipeline helpers ready for {CAMPAIGN_LABEL} at {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Pipeline helpers ready for Campaign_2 at 2026-08-27 15:39


In [4]:
# Canonical KPI extraction override: current simulations keep derived KPIs in
# summary_stats while projected_totals_stats holds additive process totals.
def extract_summary_stats(obj, label: str) -> dict:
    if not hasattr(obj, "projected_totals_stats") or not isinstance(obj.projected_totals_stats, dict) or not obj.projected_totals_stats:
        return {}
    row = dict(obj.projected_totals_stats)
    if hasattr(obj, "summary_stats") and isinstance(obj.summary_stats, dict):
        row.update(obj.summary_stats)
    if hasattr(obj, "simulation_type"):
        row["simulation_type"] = obj.simulation_type
    if hasattr(obj, "timestamp"):
        row["timestamp"] = obj.timestamp
    return {key: normalize_summary_value(value) for key, value in row.items()}

In [5]:
# Run unified pipeline once; all downstream cells reuse these variables
EXPORT_SUMMARY_TABLE = True

pipeline = run_campaign_pipeline(PIPELINE_CONFIG)

all_pkl_files = pipeline["all_pkl_files"]
candidate_files = pipeline["candidate_files"]
sweep_candidate_files = pipeline["sweep_candidate_files"]
selected_files = pipeline["selected_files"]
metadata_by_path = pipeline["metadata_by_path"]
loaded = pipeline["loaded"]
errors = pipeline["errors"]
records = pipeline["records"]
df_all = pipeline["df_all"]
df_stats = pipeline["df_stats"]

campaign_summary_table = build_campaign_summary_table(df_stats, CAMPAIGN_LABEL) if not df_stats.empty else pd.DataFrame()

sweep_counts = (
    pd.Series([metadata_by_path[p]["sweep"] for p in candidate_files], name="count")
    .value_counts()
    .rename_axis("sweep")
    .reset_index()
)

unknown_sweep_df = pd.DataFrame([
    {
        "sweep": metadata_by_path[p]["sweep"],
        "raw_sweep_token": metadata_by_path[p]["raw_sweep_token"],
        "relative_path": metadata_by_path[p]["relative_path"],
    }
    for p in candidate_files
    if metadata_by_path[p]["sweep"] == "unknown"
])

print(f"Campaign: {CAMPAIGN_LABEL}")
print(f"Configured sweep filter: {PIPELINE_CONFIG.get('sweep_filter')!r}")
print(f"Pickles discovered: {len(all_pkl_files)}")
print(f"Pickles after path filters: {len(candidate_files)}")
print(f"Pickles matching sweep: {len(sweep_candidate_files)}")
print(f"Pickles selected: {len(selected_files)}")
print(f"Loaded objects: {len(loaded)}")
print(f"Record rows: {len(records)}")
print(f"df_stats shape: {df_stats.shape}")
print(f"Summary rows: {campaign_summary_table.shape[0]}")
print("\nDetected sweep categories before selection:")
display(sweep_counts)

if not unknown_sweep_df.empty:
    print(f"\nFiles without a detected sweep token: {len(unknown_sweep_df)}")
    display(unknown_sweep_df)

if errors:
    print(f"\nLoad errors: {len(errors)}")
    for label, error in errors.items():
        print(f"  {label}: {error}")

display(campaign_summary_table)

if EXPORT_SUMMARY_TABLE and not campaign_summary_table.empty:
    out_csv = NOTEBOOK_DIR / f"{CAMPAIGN_LABEL.lower()}_kpi_mode_summary.csv"
    campaign_summary_table.to_csv(out_csv, index=False)
    print(f"Exported summary table: {out_csv}")

Campaign: Campaign_2
Configured sweep filter: 'p2'
Pickles discovered: 34
Pickles after path filters: 34
Pickles matching sweep: 34
Pickles selected: 17
Loaded objects: 17
Record rows: 17
df_stats shape: (17, 82)
Summary rows: 30

Detected sweep categories before selection:


,sweep,count
0,p2,34


,campaign,mode_group,kpi,min_value,max_value,mean_value,n_runs,kpi_label
0,Campaign_2,FedBatch,STY_kg_m3_day,0.172685,0.522024,0.388415,5,STY (kg/m3/day)
1,Campaign_2,Batch,STY_kg_m3_day,0.593106,0.593106,0.593106,1,STY (kg/m3/day)
2,Campaign_2,Chemostat,STY_kg_m3_day,1.042384,1.042384,1.042384,1,STY (kg/m3/day)
3,Campaign_2,Concentrated FedBatch,STY_kg_m3_day,2.634573,2.982897,2.791670,5,STY (kg/m3/day)
4,Campaign_2,Continuous Perfusion,STY_kg_m3_day,4.554433,5.904824,5.266744,5,STY (kg/m3/day)
5,Campaign_2,FedBatch,feed_conversion_efficiency_kg_kg,4.626801,12.996741,8.153900,5,Feed Conversion (kg feed/kg DCM)
6,Campaign_2,Batch,feed_conversion_efficiency_kg_kg,4.249168,4.249168,4.249168,1,Feed Conversion (kg feed/kg DCM)
7,Campaign_2,Chemostat,feed_conversion_efficiency_kg_kg,5.313766,5.313766,5.313766,1,Feed Conversion (kg feed/kg DCM)
8,Campaign_2,Concentrated FedBatch,feed_conversion_efficiency_kg_kg,5.248825,7.647596,6.376127,5,Feed Conversion (kg feed/kg DCM)
9,Campaign_2,Continuous Perfusion,feed_conversion_efficiency_kg_kg,4.295827,7.997261,5.845128,5,Feed Conversion (kg feed/kg DCM)


Exported summary table: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_2_fixed_stoichiometry_GFI\06_Analysis\campaign_2_kpi_mode_summary.csv


In [6]:
# Record optimization-scoring provenance separately from physical KPI aggregation.
# Objective values are only comparable for runs sharing the same scoring profile and mode.
def extract_optimization_scoring(simulation_obj) -> dict:
    manager = getattr(simulation_obj, "optimization_manager", None)
    scoring = getattr(manager, "optimization_scoring", None)
    if not isinstance(scoring, dict):
        return {
            "scoring_profile": "legacy_or_unrecorded",
            "scoring_mode": "unrecorded",
            "optimization_objective_value": np.nan,
        }

    objective_value = np.nan
    result = getattr(manager, "optimization_result", None)
    if result is not None:
        try:
            candidate_value = -float(result.fun)
            if np.isfinite(candidate_value):
                objective_value = candidate_value
        except (AttributeError, TypeError, ValueError):
            pass

    return {
        "scoring_profile": scoring.get("profile") or "unnamed_profile",
        "scoring_mode": scoring.get("mode", "unrecorded"),
        "optimization_objective_value": objective_value,
    }

if not df_stats.empty:
    scoring_df = pd.DataFrame.from_dict(
        {label: extract_optimization_scoring(obj) for label, obj in loaded.items()},
        orient="index",
    )
    scoring_df.index.name = "run"
    df_all = df_all.join(scoring_df, how="left")
    df_stats = df_stats.join(scoring_df, how="left")
    score_profiles = df_stats[["scoring_profile", "scoring_mode"]].drop_duplicates()
    if len(score_profiles) > 1:
        print(
            "⚠ Optimization objective values use different scoring profiles/modes and "
            "must not be ranked or averaged across those groups."
        )

display(df_stats[[
    column for column in (
        "run_id", "mode", "scoring_profile", "scoring_mode", "optimization_objective_value"
    ) if column in df_stats.columns
]])

,run_id,mode,scoring_profile,scoring_mode,optimization_objective_value
run,,,,,
01_FedBatch / fedbatch_constant_p2,fedbatch_constant,FedBatch,desirability,engineering_desirability,3.344864
01_FedBatch / fedbatch_csfr_p2,fedbatch_csfr,FedBatch,desirability,engineering_desirability,2.965344
01_FedBatch / fedbatch_exponential_p2,fedbatch_exponential,FedBatch,desirability,engineering_desirability,3.282807
01_FedBatch / fedbatch_gsp_p2,fedbatch_gsp,FedBatch,desirability,engineering_desirability,3.125017
01_FedBatch / fedbatch_linear_p2,fedbatch_linear,FedBatch,desirability,engineering_desirability,3.354435
02_Batch / batch_p2,batch,Batch,desirability,engineering_desirability,3.464389
03_Chemostat / chemostat_p2,chemostat,Chemostat,desirability,engineering_desirability,3.664697
04_ConcentratedFedBatch / cf_constant_p2,cf_constant,Concentrated FedBatch,desirability,engineering_desirability,4.079466
04_ConcentratedFedBatch / cf_cspr_p2,cf_cspr,Concentrated FedBatch,desirability,engineering_desirability,4.064253


## 2. Data Discovery and Loading

### 2.1 Inspect selected simulation files

In [7]:
# Inspect files discovered, sweep-filtered, and selected by the unified pipeline
print(f"Found {len(all_pkl_files)} pickle file(s) in campaign scope.")
print(f"Matched {len(sweep_candidate_files)} file(s) for sweep filter {PIPELINE_CONFIG.get('sweep_filter')!r}.")
print(f"Selected {len(selected_files)} file(s) after latest-file selection:\n")

selection_rows = []
selected_set = set(selected_files)
for p in sweep_candidate_files:
    metadata = metadata_by_path[p]
    selection_rows.append({
        "selected": p in selected_set,
        "mode": metadata["mode"],
        "mode_folder": metadata["mode_folder"],
        "run_key": metadata["run_key"],
        "sweep": metadata["sweep"],
        "export_timestamp": metadata["export_timestamp"],
        "export_folder": metadata["export_folder"],
        "relative_path": metadata["relative_path"],
    })

selection_df = pd.DataFrame(selection_rows).sort_values(
    ["selected", "mode", "run_key", "export_timestamp"],
    ascending=[False, True, True, True],
) if selection_rows else pd.DataFrame()

display(selection_df)

if not selection_df.empty:
    print("Selected files:")
    for row in selection_df.loc[selection_df["selected"]].itertuples(index=False):
        print(f"  [{row.sweep}] {row.relative_path}")

Found 34 pickle file(s) in campaign scope.
Matched 34 file(s) for sweep filter 'p2'.
Selected 17 file(s) after latest-file selection:



,selected,mode,mode_folder,run_key,sweep,export_timestamp,export_folder,relative_path
11,True,Batch,02_Batch,batch,p2,20260825_1117,batch_p2_20260825_1117,02_Batch/export/batch_p2_20260825_1117/opt_202...
13,True,Chemostat,03_Chemostat,chemostat,p2,20260825_1144,chemostat_p2_20260825_1144,03_Chemostat/export/chemostat_p2_20260825_1144...
15,True,Concentrated FedBatch,04_ConcentratedFedBatch,cf_constant,p2,20260825_1243,cf_constant_p2_20260825_1243,04_ConcentratedFedBatch/export/cf_constant_p2_...
17,True,Concentrated FedBatch,04_ConcentratedFedBatch,cf_cspr,p2,20260825_1401,cf_cspr_p2_20260825_1401,04_ConcentratedFedBatch/export/cf_cspr_p2_2026...
19,True,Concentrated FedBatch,04_ConcentratedFedBatch,cf_gsp,p2,20260825_1945,cf_gsp_p2_20260825_1945,04_ConcentratedFedBatch/export/cf_gsp_p2_20260...
21,True,Concentrated FedBatch,04_ConcentratedFedBatch,cf_tbp,p2,20260825_2111,cf_tbp_p2_20260825_2111,04_ConcentratedFedBatch/export/cf_tbp_p2_20260...
23,True,Concentrated FedBatch,04_ConcentratedFedBatch,cf_tbs,p2,20260825_2157,cf_tbs_p2_20260825_2157,04_ConcentratedFedBatch/export/cf_tbs_p2_20260...
25,True,Continuous Perfusion,05_ContinuousPerfusion,cp_constant,p2,20260825_2241,cp_constant_p2_20260825_2241,05_ContinuousPerfusion/export/cp_constant_p2_2...
27,True,Continuous Perfusion,05_ContinuousPerfusion,cp_cspr,p2,20260825_2322,cp_cspr_p2_20260825_2322,05_ContinuousPerfusion/export/cp_cspr_p2_20260...
29,True,Continuous Perfusion,05_ContinuousPerfusion,cp_gsp,p2,20260826_0306,cp_gsp_p2_20260826_0306,05_ContinuousPerfusion/export/cp_gsp_p2_202608...


Selected files:
  [p2] 02_Batch/export/batch_p2_20260825_1117/opt_20260825_1117.pkl
  [p2] 03_Chemostat/export/chemostat_p2_20260825_1144/opt_20260825_1144.pkl
  [p2] 04_ConcentratedFedBatch/export/cf_constant_p2_20260825_1243/opt_20260825_1243.pkl
  [p2] 04_ConcentratedFedBatch/export/cf_cspr_p2_20260825_1401/opt_20260825_1401.pkl
  [p2] 04_ConcentratedFedBatch/export/cf_gsp_p2_20260825_1945/opt_20260825_1945.pkl
  [p2] 04_ConcentratedFedBatch/export/cf_tbp_p2_20260825_2111/opt_20260825_2111.pkl
  [p2] 04_ConcentratedFedBatch/export/cf_tbs_p2_20260825_2157/opt_20260825_2157.pkl
  [p2] 05_ContinuousPerfusion/export/cp_constant_p2_20260825_2241/opt_20260825_2241.pkl
  [p2] 05_ContinuousPerfusion/export/cp_cspr_p2_20260825_2322/opt_20260825_2322.pkl
  [p2] 05_ContinuousPerfusion/export/cp_gsp_p2_20260826_0306/opt_20260826_0306.pkl
  [p2] 05_ContinuousPerfusion/export/cp_tbp_p2_20260826_0339/opt_20260826_0339.pkl
  [p2] 05_ContinuousPerfusion/export/cp_tbs_p2_20260826_0403/opt_20260826_04

### 2.2 Review pipeline filter settings

Adjust `PIPELINE_CONFIG` in the setup section, then re-run the pipeline-loading cell and the downstream analysis cells.

In [8]:
# Pipeline filtering/loading behavior is controlled centrally in PIPELINE_CONFIG
print("Current pipeline settings:")
print(f"  sweep_filter:             {PIPELINE_CONFIG.get('sweep_filter', 'all')!r}")
print(f"  keep_latest_only:         {PIPELINE_CONFIG.get('keep_latest_only', True)}")
print(f"  latest_scope:             {PIPELINE_CONFIG.get('latest_scope', 'per_run')}")
print(f"  skip_sensitivity_pickles: {PIPELINE_CONFIG.get('skip_sensitivity_pickles', True)}")
print(f"  exclude_patterns:         {PIPELINE_CONFIG.get('exclude_patterns', [])}")

print("\nSweep filters are detected from path tokens automatically.")
print("Use any detected token, such as 's1', 's2', or 'p2'; use 'all' or None for no filtering.")
print("Legacy '_opt_' paths are normalized to 'undefined'.")
print("To change the filter, edit PIPELINE_CONFIG in the setup cell and re-run the pipeline cell.")

Current pipeline settings:
  sweep_filter:             'p2'
  keep_latest_only:         True
  latest_scope:             per_run
  skip_sensitivity_pickles: True
  exclude_patterns:         []

Sweep filters are detected from path tokens automatically.
Use any detected token, such as 's1', 's2', or 'p2'; use 'all' or None for no filtering.
Legacy '_opt_' paths are normalized to 'undefined'.
To change the filter, edit PIPELINE_CONFIG in the setup cell and re-run the pipeline cell.


## 3. Data-Quality and Table Review

### 3.1 Verify extracted summary statistics

In [9]:
# Extraction already happened inside run_campaign_pipeline(...)
print(f"Extracted stats for {len(records)} run(s).")

if records:
    first_label = next(iter(records))
    print(f"Example run: {first_label}")
    print(f"Fields extracted: {len(records[first_label])}")

Extracted stats for 17 run(s).
Example run: 01_FedBatch / fedbatch_constant_p2
Fields extracted: 82


### 3.2 Review the run-level comparison table

In [10]:
# Reuse df_stats created by the unified pipeline
NUMERIC_COLS = [c for c in MAIN_KPIS if c in df_stats.columns]

present_numeric = list(dict.fromkeys(NUMERIC_COLS))

print(f"Comparison DataFrame: {df_stats.shape[0]} rows x {df_stats.shape[1]} columns")
print(f"Numeric KPI columns present: {present_numeric}")
df_stats

Comparison DataFrame: 17 rows x 85 columns
Numeric KPI columns present: ['STY_kg_m3_day', 'total_DCM_harvested_kg', 'medium_conversion_efficiency_m3_kg', 'feed_conversion_efficiency_kg_kg', 'protein_conversion_efficiency_kg_kg', 'number_of_batches']


,simulation_type,timestamp,mode,run_id,sweep,run_key,export_folder,export_timestamp,STY_kg_m3_day,total_DCM_harvested_kg,...,total_nutrient_consumed_kg,total_nutrient_in_kg,total_nutrient_out_kg,total_outlet_volume_L,total_permeate_volume_L,total_water_produced_kg,total_water_produced_mmol,scoring_profile,scoring_mode,optimization_objective_value
run,,,,,,,,,,,,,,,,,,,,,
01_FedBatch / fedbatch_constant_p2,Fedbatch,2026-08-25T00:02:40.294254,FedBatch,fedbatch_constant,p2,fedbatch_constant,fedbatch_constant_p2_20260825_0002,20260825_0002,0.522024,334.095163,...,1004.816703,2025.457137,1020.640434,1.071934e+05,0.000000e+00,261.049363,1.449044e+07,desirability,engineering_desirability,3.344864
01_FedBatch / fedbatch_csfr_p2,Fedbatch,2026-08-25T00:50:37.742467,FedBatch,fedbatch_csfr,p2,fedbatch_csfr,fedbatch_csfr_p2_20260825_0050,20260825_0050,0.172685,110.518664,...,331.522930,1259.825026,928.302096,3.092769e+04,0.000000e+00,86.129953,4.780939e+06,desirability,engineering_desirability,2.965344
01_FedBatch / fedbatch_exponential_p2,Fedbatch,2026-08-25T01:45:40.988100,FedBatch,fedbatch_exponential,p2,fedbatch_exponential,fedbatch_exponential_p2_20260825_0145,20260825_0145,0.372704,238.530527,...,717.026743,1103.633180,386.606436,7.659340e+04,0.000000e+00,186.280794,1.034016e+07,desirability,engineering_desirability,3.282807
01_FedBatch / fedbatch_gsp_p2,Fedbatch,2026-08-25T09:25:16.831145,FedBatch,fedbatch_gsp,p2,fedbatch_gsp,fedbatch_gsp_p2_20260825_0925,20260825_0925,0.357630,228.883379,...,687.130036,2974.737997,2287.607961,9.197440e+04,0.000000e+00,178.731531,9.921108e+06,desirability,engineering_desirability,3.125017
01_FedBatch / fedbatch_linear_p2,Fedbatch,2026-08-25T10:21:19.571533,FedBatch,fedbatch_linear,p2,fedbatch_linear,fedbatch_linear_p2_20260825_1021,20260825_1021,0.517032,330.900242,...,995.195142,1880.915339,885.720197,1.071226e+05,0.000000e+00,258.551559,1.435179e+07,desirability,engineering_desirability,3.354435
02_Batch / batch_p2,Batch (startup Fedbatch),2026-08-25T11:17:40.139504,Batch,batch,p2,batch,batch_p2_20260825_1117,20260825_1117,0.593106,379.587969,...,1141.817476,1612.933234,471.115758,1.072398e+05,0.000000e+00,296.633878,1.646568e+07,desirability,engineering_desirability,3.464389
03_Chemostat / chemostat_p2,Chemostat,2026-08-25T11:44:14.053323,Chemostat,chemostat,p2,chemostat,chemostat_p2_20260825_1144,20260825_1144,1.042384,667.125544,...,2007.722082,3544.948973,1537.226891,2.483178e+05,0.000000e+00,521.609951,2.895375e+07,desirability,engineering_desirability,3.664697
04_ConcentratedFedBatch / cf_constant_p2,Concentrated Fedbatch,2026-08-25T12:43:46.689448,Concentrated FedBatch,cf_constant,p2,cf_constant,cf_constant_p2_20260825_1243,20260825_1243,2.951712,1889.095418,...,5983.407401,11434.738571,5451.331171,1.069377e+06,9.773772e+05,1554.527792,8.628940e+07,desirability,engineering_desirability,4.079466
04_ConcentratedFedBatch / cf_cspr_p2,Concentrated Fedbatch,2026-08-25T14:01:14.681772,Concentrated FedBatch,cf_cspr,p2,cf_cspr,cf_cspr_p2_20260825_1401,20260825_1401,2.634851,1686.304341,...,5387.902191,8851.116264,3463.214073,9.247336e+05,8.327336e+05,1399.791858,7.770026e+07,desirability,engineering_desirability,4.064253


In [11]:
# ── Styled view: highlight best value in each numeric column ─────────────────
from animakinetica.core.simulation_data import SUMMARY_METRIC_DIRECTIONS


def highlight_best(series):
    """Highlight the direction-aware best finite value in a KPI column."""
    direction = SUMMARY_METRIC_DIRECTIONS.get(series.name, "maximize")
    numeric_series = pd.to_numeric(series, errors="coerce")

    if numeric_series.notna().any():
        best_value = (
            numeric_series.min()
            if direction == "minimize"
            else numeric_series.max()
        )
        is_best = numeric_series.eq(best_value)
    else:
        is_best = pd.Series(False, index=series.index)

    color = "#d4edda" #if direction == "maximize" else "#cfe2ff"
    return ["background-color: {}; font-weight: bold".format(color) if value else "" for value in is_best]


minimize_kpis = [
    kpi for kpi in present_numeric
    if SUMMARY_METRIC_DIRECTIONS.get(kpi, "maximize") == "minimize"
]

styled = (
    df_stats[present_numeric + ["mode"]]
    .style
    .apply(highlight_best, subset=present_numeric)
    .format({column: "{:.4g}" for column in present_numeric})
    .set_caption(
        "Summary Statistics — best value per column highlighted "
        "(green: maximize; blue: minimize). "
        f"Minimize KPIs: {', '.join(minimize_kpis) or 'none'}"
    )
)
styled

,STY_kg_m3_day,total_DCM_harvested_kg,medium_conversion_efficiency_m3_kg,feed_conversion_efficiency_kg_kg,protein_conversion_efficiency_kg_kg,number_of_batches,mode
run,,,,,,,
01_FedBatch / fedbatch_constant_p2,0.522,334.1,0.3208,6.063,5.618,56,FedBatch
01_FedBatch / fedbatch_csfr_p2,0.1727,110.5,0.2798,11.4,8.824,16,FedBatch
01_FedBatch / fedbatch_exponential_p2,0.3727,238.5,0.3211,4.627,3.949,40,FedBatch
01_FedBatch / fedbatch_gsp_p2,0.3576,228.9,0.4017,13,6.071,48,FedBatch
01_FedBatch / fedbatch_linear_p2,0.517,330.9,0.3237,5.684,5.047,56,FedBatch
02_Batch / batch_p2,0.5931,379.6,0.2824,4.249,1.961,56,Batch
03_Chemostat / chemostat_p2,1.042,667.1,0.3722,5.314,3.232,8,Chemostat
04_ConcentratedFedBatch / cf_constant_p2,2.952,1889,0.5647,6.053,4.185,48,Concentrated FedBatch
04_ConcentratedFedBatch / cf_cspr_p2,2.635,1686,0.5474,5.249,3.248,48,Concentrated FedBatch


In [12]:
# ── Optional CSV export ───────────────────────────────────────────────────────
EXPORT_CSV = True   # set to False to skip

if EXPORT_CSV:
    csv_path = NOTEBOOK_DIR / "summary_comparison.csv"
    df_stats.to_csv(csv_path)
    print(f"Exported to: {csv_path}")

Exported to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_2_fixed_stoichiometry_GFI\06_Analysis\summary_comparison.csv


## 4. Run-Level Overview Figures

These figures compare the selected optimized runs directly. Bar plots are grouped by operating mode; the heatmap and radar charts normalise values only for visual comparison.

In [13]:
# Shared plotting data initialization (consistent ordering and labels for all figures)
import plotly.express as px

def short_label(full_label: str) -> str:
    # Labels now end in <run_key>_<sweep>; preserve the sweep in display labels.
    return str(full_label).split(" / ")[-1]

mode_rank_map = {canonical_mode(m): i for i, m in enumerate(MODE_ORDER)}
def appearance_rank(label: str) -> int:
    cm = canonical_mode(label)
    return mode_rank_map.get(cm, len(mode_rank_map))

plot_data_ordered = df_stats.copy()
plot_data_ordered["short_label"] = [short_label(lbl) for lbl in plot_data_ordered.index]
plot_data_ordered["appearance_rank"] = plot_data_ordered["mode"].map(mode_rank_map).fillna(len(mode_rank_map))
plot_data_ordered = plot_data_ordered.sort_values(["appearance_rank", "short_label"]).reset_index(drop=True)
category_order_vertical = plot_data_ordered["short_label"].tolist()
category_order_horizontal = plot_data_ordered["short_label"].tolist()[::-1]
mode_order = list(dict.fromkeys(plot_data_ordered["mode"].tolist()))
color_seq = px.colors.qualitative.Set2
mode_colors = {m: color_seq[i % len(color_seq)] for i, m in enumerate(mode_order)}
print("Ordered run labels:")
for i, lbl in enumerate(category_order_vertical, start=1):
    print(f"  {i}. {lbl}")

Ordered run labels:
  1. fedbatch_constant_p2
  2. fedbatch_csfr_p2
  3. fedbatch_exponential_p2
  4. fedbatch_gsp_p2
  5. fedbatch_linear_p2
  6. batch_p2
  7. chemostat_p2
  8. cf_constant_p2
  9. cf_cspr_p2
  10. cf_gsp_p2
  11. cf_tbp_p2
  12. cf_tbs_p2
  13. cp_constant_p2
  14. cp_cspr_p2
  15. cp_gsp_p2
  16. cp_tbp_p2
  17. cp_tbs_p2


In [14]:
# ── KPI Bar Plots: one subplot per KPI, grouped by mode ───────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots


plot_cols = [c for c in KPI_LABELS_MAIN if c in df_stats.columns]
if not plot_cols:
    raise ValueError("No KPI columns available to plot.")

n_plots = len(plot_cols)
fig = make_subplots(
    rows=n_plots,
    cols=1,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.09,
 )

for row, col in enumerate(plot_cols, start=1):
    for mode in mode_order:
        sub = plot_data_ordered.loc[plot_data_ordered["mode"] == mode, ["short_label", col]].copy()
        sub[col] = sub[col].astype(float)

        fig.add_trace(
            go.Bar(
                x=sub["short_label"],
                y=sub[col],
                name=mode,
                marker_color=mode_colors[mode],
                text=[f"{v:.3g}" for v in sub[col]],
                textposition="outside",
                cliponaxis=False,
                showlegend=(row == 1),
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    + f"{KPI_LABELS_MAIN[col]}: "
                    + "%{y:.4g}<br>"
                    + f"Mode: {mode}<extra></extra>"
                ),
            ),
            row=row,
            col=1,
        )

    fig.update_xaxes(
        categoryorder="array",
        categoryarray=category_order_vertical,
        row=row,
        col=1,
    )
    fig.update_yaxes(
        title_text=KPI_LABELS_MAIN[col],
        row=row,
        col=1,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.10)",
    )

fig.update_layout(
    height=max(700, 340 * n_plots),
    width=1100,
    barmode="group",
    bargap=0.25,
    legend_title_text="Operating Mode",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(l=80, r=20, t=140, b=20),
    template="plotly_white",
)

fig.show()


In [15]:
SAVEFIG = False

if SAVEFIG:
    out_html = NOTEBOOK_DIR / "summary_bar_plots_plotly.html"
    fig.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html.name}")

### 4.2 Horizontal KPI bar charts

In [16]:
# Alternative horizontal orientation for better label readability (if labels are long)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if "plot_cols" not in globals() or not plot_cols:
    raise ValueError("No KPI columns available to plot.")

n_plots = len(plot_cols)
fig = make_subplots(
    rows=n_plots,
    cols=1,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.04,
)

for row, col in enumerate(plot_cols, start=1):
    for mode in mode_order:
        sub = plot_data_ordered.loc[plot_data_ordered["mode"] == mode, ["short_label", col]].copy()
        sub[col] = sub[col].astype(float)

        fig.add_trace(
            go.Bar(
                x=sub[col],
                y=sub["short_label"],
                orientation="h",
                name=mode,
                marker_color=mode_colors[mode],
                text=[f"{v:.3g}" for v in sub[col]],
                textposition="outside",
                cliponaxis=False,
                showlegend=(row == 1),
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    + f"{KPI_LABELS_MAIN[col]}: "
                    + "%{x:.4g}<br>"
                    + f"Mode: {mode}<extra></extra>"
                ),
            ),
            row=row,
            col=1,
        )

    fig.update_xaxes(
        title_text="",
        row=row,
        col=1,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.10)",
    )
    fig.update_yaxes(
        title_text=KPI_LABELS_MAIN[col],
        title_standoff=18,
        row=row,
        col=1,
        categoryorder="array",
        categoryarray=category_order_horizontal,
        automargin=True,
        tickfont=dict(size=9),
        title_font=dict(size=10),
    )

fig.update_layout(
    height=max(480, 200 * n_plots + 20 * len(plot_data_ordered)),
    width=1000,
    bargap=0.2,
    legend_title_text="Operating Mode",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.01,
        xanchor="center",
        x=0.4,
    ),
    margin=dict(l=180, r=20, t=20, b=20),
    template="plotly_white",
)

fig.show()


In [17]:
SAVEFIG = False

if SAVEFIG:
    out_html = NOTEBOOK_DIR / "summary_horizontal_bars_campaign1.html"
    fig.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html.name}")

In [18]:
# Heatmap of normalized KPI values across runs, with original values as annotations
import plotly.graph_objects as go

if "plot_cols" not in globals() or not plot_cols:
    raise ValueError("No KPI columns available to plot.")

# Normalized heatmap using a zero baseline and the observed column maximum
heat_data = plot_data_ordered.set_index("short_label")[plot_cols].astype(float)
heat_norm = heat_data.divide(heat_data.max().replace(0, 1), axis=1)
heat_norm.columns = [KPI_LABELS_MAIN.get(c, c) for c in heat_norm.columns]

# Annotate with original (non-normalized) values
text_vals = heat_data.rename(columns=KPI_LABELS_MAIN).map(lambda v: f"{v:.3g}").values

fig2 = go.Figure(
    data=go.Heatmap(
        z=heat_norm.values,
        x=list(heat_norm.columns),
        y=list(heat_norm.index),
        text=text_vals,
        texttemplate="%{text}",
        textfont={"size": 9},
        colorscale="YlGn",
        zmin=0.0,
        zmax=1.0,
        colorbar={"title": "Normalised to KPI max"},
        hovertemplate=(
            "<b>%{y}</b><br>"
            + "%{x}<br>"
            + "Normalised: %{z:.3f}<extra></extra>"
        ),
    )
)

fig2.update_layout(
    title=f"{CAMPAIGN_LABEL} - Performance Heatmap (Normalised to KPI Max)",
    xaxis_title="",
    width=max(900, 220 + 170 * len(heat_norm.columns)),
    height=max(420, 120 + 45 * len(heat_norm.index)),
    margin=dict(l=160, r=40, t=70, b=70),
    template="plotly_white",
)

fig2.update_xaxes(tickangle=30)

fig2.show()

In [19]:
SAVEFIG = False

if SAVEFIG:
    out_html = NOTEBOOK_DIR / "summary_heatmap_plotly.html"
    fig2.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html.name}")

### 4.4 Operating-mode radar charts

Normalised KPI profiles are averaged within each operating mode for display.

In [20]:
# Spider/radar charts of normalised KPI performance profiles per operating mode
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if "plot_data_ordered" not in globals() or "plot_cols" not in globals() or not plot_cols:
    raise ValueError("Run the shared plotting data initialization and KPI plotting cells first.")

# Use the same ordered data used by all figures, then normalize KPI columns to [0, 1]
radar_source = plot_data_ordered[["mode"] + plot_cols].copy()
radar_vals = radar_source[plot_cols].astype(float)
radar_norm = (radar_vals - radar_vals.min()) / (radar_vals.max() - radar_vals.min()).replace(0, 1)
radar_source_norm = pd.concat([radar_source[["mode"]], radar_norm], axis=1)

# Aggregate normalized performance per operating mode (mean over runs per mode)
mode_profiles = radar_source_norm.groupby("mode", as_index=True)[plot_cols].mean()

# Keep mode and color conventions consistent with other figures
modes = [m for m in mode_order if m in mode_profiles.index]
categories = [KPI_LABELS_MAIN.get(c, c) for c in plot_cols]

# Build one spider chart per operating mode
n_modes = len(modes)
n_cols = 3
n_rows = math.ceil(n_modes / n_cols)

fig_spider = make_subplots(
    rows=n_rows,
    cols=n_cols,
    specs=[[{"type": "polar"} for _ in range(n_cols)] for _ in range(n_rows)],
    subplot_titles=modes,
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
 )

for i, mode in enumerate(modes):
    r = mode_profiles.loc[mode, plot_cols].tolist()
    r_closed = r + [r[0]]
    theta_closed = categories + [categories[0]]

    row = (i // n_cols) + 1
    col = (i % n_cols) + 1

    fig_spider.add_trace(
        go.Scatterpolar(
            r=r_closed,
            theta=theta_closed,
            mode="lines+markers",
            fill="toself",
            name=mode,
            line=dict(color=mode_colors.get(mode, "#1f77b4"), width=2),
            marker=dict(size=5),
            opacity=0.75,
            showlegend=True,
        ),
        row=row,
        col=col,
    )

# Apply common radial axis settings to all polar subplots
for i in range(1, n_rows * n_cols + 1):
    polar_key = "polar" if i == 1 else f"polar{i}"
    fig_spider.update_layout(
        **{
            polar_key: dict(
                radialaxis=dict(
                    range=[0, 1],
                    tickmode="array",
                    tickvals=[0.0, 0.25, 0.5, 0.75, 1.0],
                    tickfont=dict(size=10),
                ),
                angularaxis=dict(
                    direction="clockwise",
                    rotation=55,
                    tickfont=dict(size=8),
                ),
            )
        }
    )

# Make subplot titles slightly smaller to avoid overlap
for ann in fig_spider.layout.annotations:
    ann.font.size = 11

fig_spider.update_layout(
    # title=dict(
    #     text=f"{CAMPAIGN_LABEL} - Spider Charts of Normalised KPI Performance",
    #     x=0.5,
    #     xanchor="center",
    #     y=0.995,
    #     yanchor="top",
    # ),
    height=max(520, 300 * n_rows),
    width=1450,
    legend_title_text="Operating Mode",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.06,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(l=60, r=20, t=80, b=20),
    template="plotly_white",
)

fig_spider.show()

In [21]:
SAVEFIG = False

if SAVEFIG:
    out_html = NOTEBOOK_DIR / "summary_spider_charts_plotly.html"
    fig_spider.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html.name}")

## 5. Mode-Level Aggregated Figures

### 5.1 KPI ranges and means by operating mode

This lollipop/range overview uses the aggregated `campaign_summary_table`; it is distinct from the run-level heatmap above.

In [22]:
# Faceted lollipop overview of aggregated KPI ranges and means by operating mode
# Uses campaign_summary_table rather than individual run rows.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if "campaign_summary_table" not in globals():
    raise ValueError("Run the campaign summary-table cell first.")

if "mode_order" not in globals() or not mode_order:
    raise ValueError("Run the shared plotting data initialization cell first.")

plot_tbl = campaign_summary_table.copy()
if plot_tbl.empty:
    raise ValueError("campaign_summary_table is empty.")

# Optional campaign filter (kept for future multi-campaign files merged into one table)
CAMPAIGN_FILTER = [CAMPAIGN_LABEL]
if CAMPAIGN_FILTER:
    plot_tbl = plot_tbl[plot_tbl["campaign"].isin(CAMPAIGN_FILTER)].copy()

kpi_order = [k for k in MAIN_KPIS if k in plot_tbl["kpi"].unique()]
if not kpi_order:
    raise ValueError("No matching KPI rows found in campaign_summary_table.")

# Reuse the mode ordering determined in the shared plotting initialization cell
mode_group_order = []
for m in mode_order:
    cm = canonical_mode(m)
    if cm not in mode_group_order:
        mode_group_order.append(cm)
mode_group_order = [m for m in mode_group_order if m in plot_tbl["mode_group"].astype(str).unique()]
mode_to_y = {m: i for i, m in enumerate(mode_group_order)}

# Single hue for ranges and endpoints; darker markers show mode means.
BASE_COLOR = "rgba(33, 113, 181, 0.90)"
RANGE_COLOR = "rgba(33, 113, 181, 0.35)"
MEAN_COLOR = "rgba(8, 48, 107, 1.00)"

fig = make_subplots(
    rows=len(kpi_order),
    cols=1,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.1,
    #subplot_titles=[KPI_LABELS_MAIN.get(k, k) for k in kpi_order],
)

for row_idx, kpi in enumerate(kpi_order, start=1):
    sub = plot_tbl[plot_tbl["kpi"] == kpi].copy()
    sub["y"] = sub["mode_group"].astype(str).map(mode_to_y)

    # Multi-run ranges
    multi = sub[sub["n_runs"] > 1]
    for _, r in multi.iterrows():
        fig.add_trace(
            go.Scatter(
                x=[r["min_value"], r["max_value"]],
                y=[r["y"], r["y"]],
                mode="lines",
                line=dict(width=3, color=RANGE_COLOR),
                hovertemplate=(
                    f"<b>{r['campaign']}</b><br>"
                    + f"Mode: {r['mode_group']}<br>"
                    + "Range: %{x:.4g}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=row_idx,
            col=1,
        )

    # Endpoints for multi-run ranges
    if not multi.empty:
        fig.add_trace(
            go.Scatter(
                x=np.r_[multi["min_value"].values, multi["max_value"].values],
                y=np.r_[multi["y"].values, multi["y"].values],
                mode="markers",
                marker=dict(size=6, color=BASE_COLOR),
                hovertemplate="Value: %{x:.4g}<extra></extra>",
                showlegend=False,
            ),
            row=row_idx,
            col=1,
        )

    # Mean marker for multi-run modes
    if not multi.empty:
        fig.add_trace(
            go.Scatter(
                x=multi["mean_value"],
                y=multi["y"],
                mode="markers",
                marker=dict(size=9, color=MEAN_COLOR),
                hovertemplate=("<b>%{text}</b><br>Mean: %{x:.4g}<extra></extra>"),
                text=multi["mode_group"].astype(str),
                showlegend=False,
            ),
            row=row_idx,
            col=1,
        )

    # Single-run modes: one dot only
    single = sub[sub["n_runs"] == 1]
    if not single.empty:
        fig.add_trace(
            go.Scatter(
                x=single["mean_value"],
                y=single["y"],
                mode="markers",
                marker=dict(size=7, color=BASE_COLOR),
                hovertemplate=("<b>%{text}</b><br>Value: %{x:.4g}<extra></extra>"),
                text=single["mode_group"].astype(str),
                showlegend=False,
            ),
            row=row_idx,
            col=1,
        )

    fig.update_yaxes(
        tickmode="array",
        tickvals=list(mode_to_y.values()),
        ticktext=list(mode_to_y.keys()),
        autorange="reversed",
        row=row_idx,
        col=1,
    )

    # Per-subplot x-axis title (remove shared/global title approach)
    fig.update_xaxes(
        title_text=KPI_LABELS_MAIN.get(kpi, kpi),
        showgrid=True,
        gridcolor="rgba(0,0,0,0.10)",
        zeroline=False,
        row=row_idx,
        col=1,
    )

fig.update_layout(
    template="plotly_white",
    width=900,
    height=max(500, 145 * len(kpi_order)),
    showlegend=False,
    margin=dict(l=185, r=30, t=50, b=55),
)

fig.show()


In [23]:
SAVEFIG = False

if SAVEFIG:
    out_html = NOTEBOOK_DIR / f"{CAMPAIGN_LABEL.lower()}_kpi_lollipop_faceted_rows.html"
    fig.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html.name}")

### 5.2 Mean KPI heatmap by operating mode

The heatmap uses the same aggregated `campaign_summary_table` and displays mean KPI values per operating mode.

In [24]:
# Heatmap of mean KPI values per operating mode, with optional normalization and annotations
import numpy as np
import plotly.graph_objects as go

if "campaign_summary_table" not in globals():
    raise ValueError("Run the campaign summary-table cell first.")

if "mode_order" not in globals() or not mode_order:
    raise ValueError("Run the shared plotting data initialization cell first.")

# Reuse same campaign filter convention as the lollipop overview
CAMPAIGN_FILTER = [CAMPAIGN_LABEL]
plot_tbl = campaign_summary_table.copy()
if CAMPAIGN_FILTER:
    plot_tbl = plot_tbl[plot_tbl["campaign"].isin(CAMPAIGN_FILTER)].copy()

if plot_tbl.empty:
    raise ValueError("No rows available after campaign filtering.")

kpi_order = [k for k in MAIN_KPIS if k in plot_tbl["kpi"].unique()]
if not kpi_order:
    raise ValueError("No matching KPI rows found in campaign_summary_table.")

# Reuse the mode ordering determined in the shared plotting initialization cell
mode_group_order = []
for m in mode_order:
    cm = canonical_mode(m)
    if cm not in mode_group_order:
        mode_group_order.append(cm)
mode_group_order = [m for m in mode_group_order if m in plot_tbl["mode_group"].astype(str).unique()]

# Pivot means to matrix: rows = operating mode, columns = KPI
heat_df = (
    plot_tbl[plot_tbl["kpi"].isin(kpi_order)]
    .pivot_table(index="mode_group", columns="kpi", values="mean_value", aggfunc="mean", observed=False)
    .reindex(index=mode_group_order, columns=kpi_order)
)

# Format values without scientific notation
def format_no_sci(v: float, max_decimals: int = 6) -> str:
    if pd.isna(v):
        return ""
    s = np.format_float_positional(float(v), precision=max_decimals, trim='-')
    if "." in s:
        s = s.rstrip("0").rstrip(".")
    return s

# Keep annotation as original mean values (no scientific notation)
text_vals = heat_df.map(lambda v: format_no_sci(v, max_decimals=2)).values

# Normalize per KPI column against a zero baseline and the observed column maximum
NORMALIZE_WITHIN_KPI = True
if NORMALIZE_WITHIN_KPI:
    heat_for_color = heat_df.divide(heat_df.max().replace(0, 1), axis=1)
    colorbar_title = "Normalised to KPI max"
else:
    heat_for_color = heat_df.copy()
    colorbar_title = "Mean KPI value"

x_labels = [KPI_LABELS_MAIN.get(k, k) for k in heat_df.columns]
y_labels = [str(v) for v in heat_df.index]

fig_hm = go.Figure(
    data=go.Heatmap(
        z=heat_for_color.values,
        x=x_labels,
        y=y_labels,
        text=text_vals,
        texttemplate="%{text}",
        textfont={"size": 10},
        colorscale="greens",
        zmin=0.0 if NORMALIZE_WITHIN_KPI else None,
        zmax=1.0 if NORMALIZE_WITHIN_KPI else None,
        colorbar={"title": colorbar_title},
        hovertemplate=(
            "<b>%{y}</b><br>"
            + "%{x}<br>"
            + "Value: %{text}<extra></extra>"
        ),
    )
)

fig_hm.update_layout(
    template="plotly_white",
    title="Mean KPI by Operating Mode (Normalised to KPI Max)",
    width=950,
    height=max(360, 90 + 55 * len(y_labels)),
    margin=dict(l=190, r=40, t=70, b=70),
)

fig_hm.update_xaxes(tickangle=20)
fig_hm.show()

In [25]:
SAVEFIG = False

if SAVEFIG:
    out_html = NOTEBOOK_DIR / f"{CAMPAIGN_LABEL.lower()}_mean_kpi_heatmap.html"
    fig_hm.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html.name}")

## 6. Strategy Comparison Figures

These figures compare strategy-level averages within operational modes. Delta figures use the configured constant feed or perfusion strategy as their baseline.

### 6.1 Shared strategy configuration and plotting helpers

In [26]:
# Shared configuration and helpers for strategy-level heatmaps and delta-to-baseline charts
import re
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if "plot_data_ordered" not in globals() or "plot_cols" not in globals() or not plot_cols:
    raise ValueError("Run the shared plotting data initialization cell first.")

def format_no_sci(v: float, max_decimals: int = 6) -> str:
    if pd.isna(v):
        return ""
    s = np.format_float_positional(float(v), precision=max_decimals, trim="-")
    if "." in s:
        s = s.rstrip("0").rstrip(".")
    return s

def extract_strategy_label(label: str, mode_token: str, strategy_map: dict[str, str]) -> str:
    """Extract and canonicalise the strategy from a run label.

    Labels retain the selected sweep suffix (for example ``constant_p2``).
    Strip any terminal sweep token before mapping so the configured baseline
    consistently resolves to ``Constant``.
    """
    text = str(label).lower().strip()
    text = re.sub(r"^c\d+_", "", text)
    if mode_token and mode_token in text:
        text = text.split(mode_token, 1)[1]

    # Strip any terminal sweep or optimization suffix, e.g. _p2, _s1, _opt.
    text = re.sub(r"_(?:[a-z]+\d+|opt)$", "", text)
    return strategy_map.get(text, text.replace("_", " ").title())

def save_figure_svg(fig: go.Figure, output_path, width: int | None = None, height: int | None = None) -> None:
    export_fig = go.Figure(fig)
    export_fig.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
    )
    export_fig.write_image(str(output_path), format="svg", width=width, height=height)

def _placeholder_figure(title: str, message: str, width: int = 950, height: int = 380) -> go.Figure:
    fig = go.Figure()
    fig.add_annotation(
        text=message,
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=14),
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=width,
        height=height,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
    )
    return fig

STRATEGY_DELTA_KPIS = {
    "STY_kg_m3_day": "STY",
    "medium_conversion_efficiency_m3_kg": "Medium Conversion",
    "feed_conversion_efficiency_kg_kg": "Feed Conversion",
    "protein_conversion_efficiency_kg_kg": "Protein Conversion",
}

STRATEGY_COMPARISON_ORDER = [
    "fed_batch",
    "concentrated_fedbatch",
    "continuous_perfusion",
]

STRATEGY_HEATMAP_CONFIG = {
    "fed_batch": {
        "mode_name": "FedBatch",
        "mode_token": "fedbatch_",
        "strategy_column": "feed_strategy",
        "strategy_axis_title": "Feed strategy",
        "baseline_strategy": "Constant",
        "strategy_map": {
            "constant": "Constant",
            "linear": "Linear",
            "exponential": "Exponential",
            "csfr": "CSFR",
            "gsp": "Glucose Setpoint",
            "glucose_setpoint": "Glucose Setpoint",
        },
        "preferred_order": ["Constant", "Linear", "Exponential", "CSFR", "Glucose Setpoint"],
        "title": "Fed-Batch Feed Strategy Heatmap (Normalised to KPI Max)",
        "delta_title": "Fed-Batch Delta to Constant Feed Strategy",
        "delta_heatmap_title": "Fed-Batch Delta to Constant Feed Strategy (%)",
        "output_name": f"{CAMPAIGN_LABEL.lower()}_fedbatch_feed_strategy_heatmap.svg",
        "delta_output_name": f"{CAMPAIGN_LABEL.lower()}_fedbatch_delta_to_constant_feed.svg",
        "delta_heatmap_output_name": f"{CAMPAIGN_LABEL.lower()}_fedbatch_delta_heatmap.svg",
    },
    "concentrated_fedbatch": {
        "mode_name": "Concentrated FedBatch",
        "mode_token": "cf_",
        "strategy_column": "perfusion_strategy",
        "strategy_axis_title": "Perfusion strategy",
        "baseline_strategy": "Constant",
        "strategy_map": {
            "constant": "Constant",
            "cspr": "CSPR",
            "gsp": "Glucose Setpoint",
            "tbp": "Toxicity-Based Proportional",
            "tbs": "Toxicity-Based Step",
        },
        "preferred_order": ["Constant", "Toxicity-Based Step", "Toxicity-Based Proportional", "CSPR", "Glucose Setpoint"],
        "title": "Concentrated Fed-Batch Perfusion Strategy Heatmap (Normalised to KPI Max)",
        "delta_title": "Concentrated Fed-Batch Delta to Constant Perfusion Strategy",
        "delta_heatmap_title": "Concentrated Fed-Batch Delta to Constant Perfusion Strategy (%)",
        "output_name": f"{CAMPAIGN_LABEL.lower()}_concentrated_fedbatch_perfusion_strategy_heatmap.svg",
        "delta_output_name": f"{CAMPAIGN_LABEL.lower()}_concentrated_fedbatch_delta_to_constant_perfusion.svg",
        "delta_heatmap_output_name": f"{CAMPAIGN_LABEL.lower()}_concentrated_fedbatch_delta_heatmap.svg",
    },
    "continuous_perfusion": {
        "mode_name": "Continuous Perfusion",
        "mode_token": "cp_",
        "strategy_column": "perfusion_strategy",
        "strategy_axis_title": "Perfusion strategy",
        "baseline_strategy": "Constant",
        "strategy_map": {
            "constant": "Constant",
            "cspr": "CSPR",
            "gsp": "Glucose Setpoint",
            "tbp": "Toxicity-Based Proportional",
            "tbs": "Toxicity-Based Step",
        },
        "preferred_order": ["Constant", "Toxicity-Based Step", "Toxicity-Based Proportional", "CSPR", "Glucose Setpoint"],
        "title": "Continuous Perfusion Strategy Heatmap (Normalised to KPI Max)",
        "delta_title": "Continuous Perfusion Delta to Constant Perfusion Strategy",
        "delta_heatmap_title": "Continuous Perfusion Delta to Constant Perfusion Strategy (%)",
        "output_name": f"{CAMPAIGN_LABEL.lower()}_continuous_perfusion_strategy_heatmap.svg",
        "delta_output_name": f"{CAMPAIGN_LABEL.lower()}_continuous_perfusion_delta_to_constant.svg",
        "delta_heatmap_output_name": f"{CAMPAIGN_LABEL.lower()}_continuous_perfusion_delta_heatmap.svg",
    },
}

def get_strategy_matrix(config_key: str):
    cfg = STRATEGY_HEATMAP_CONFIG[config_key]
    strategy_source = plot_data_ordered.loc[
        plot_data_ordered["mode"].map(canonical_mode).eq(cfg["mode_name"])
    ].copy()
    if strategy_source.empty:
        raise ValueError(f"No rows found for {cfg['mode_name']}.")

    strategy_source[cfg["strategy_column"]] = strategy_source["short_label"].map(
        lambda label: extract_strategy_label(label, cfg["mode_token"], cfg["strategy_map"])
    )
    strategy_kpis = [col for col in plot_cols if col in strategy_source.columns]
    if not strategy_kpis:
        raise ValueError(f"No KPI columns available for {cfg['mode_name']}.")

    strategy_raw = (
        strategy_source.groupby(cfg["strategy_column"], as_index=True)[strategy_kpis]
        .mean()
        .astype(float)
    )
    strategy_order = [s for s in cfg["preferred_order"] if s in strategy_raw.index]
    strategy_order += [s for s in strategy_raw.index if s not in strategy_order]
    strategy_raw = strategy_raw.reindex(index=strategy_order, columns=strategy_kpis)
    return cfg, strategy_raw

def get_strategy_delta_data(config_key: str):
    cfg, strategy_raw = get_strategy_matrix(config_key)
    delta_kpis = [k for k in STRATEGY_DELTA_KPIS if k in strategy_raw.columns]
    if not delta_kpis:
        raise ValueError(f"No configured delta KPIs are available for {cfg['mode_name']}.")

    missing = [k for k in STRATEGY_DELTA_KPIS if k not in delta_kpis]
    if missing:
        print(f"{cfg['mode_name']}: skipping unavailable delta KPIs: {missing}")

    baseline_strategy = cfg["baseline_strategy"]
    if baseline_strategy not in strategy_raw.index:
        available = ", ".join(map(str, strategy_raw.index))
        raise ValueError(
            f"Baseline strategy '{baseline_strategy}' not found for {cfg['mode_name']}. "
            f"Available strategies: {available}."
        )

    baseline = strategy_raw.loc[baseline_strategy, delta_kpis]
    delta_pct = strategy_raw[delta_kpis].subtract(baseline, axis=1).divide(baseline.replace(0, np.nan), axis=1) * 100
    delta_pct = delta_pct.fillna(0.0)

    plotted_delta_pct = delta_pct.drop(index=baseline_strategy, errors="ignore")
    plotted_raw = strategy_raw.loc[plotted_delta_pct.index, delta_kpis]
    return cfg, plotted_delta_pct, plotted_raw, baseline

def build_strategy_heatmap(config_key: str):
    cfg = STRATEGY_HEATMAP_CONFIG[config_key]
    try:
        cfg, strategy_raw = get_strategy_matrix(config_key)
    except ValueError as exc:
        fig = _placeholder_figure(cfg["title"], str(exc))
        return fig, pd.DataFrame(), pd.DataFrame()

    strategy_norm = strategy_raw.divide(strategy_raw.max().replace(0, 1), axis=1)
    strategy_text = strategy_raw.map(lambda v: format_no_sci(v, max_decimals=2)).values
    strategy_x = [KPI_LABELS_MAIN.get(k, k) for k in strategy_raw.columns]
    strategy_y = [str(v) for v in strategy_raw.index]

    fig = go.Figure(
        data=go.Heatmap(
            z=strategy_norm.values,
            x=strategy_x,
            y=strategy_y,
            text=strategy_text,
            texttemplate="%{text}",
            textfont={"size": 10},
            colorscale="greens",
            zmin=0.0,
            zmax=1.0,
            colorbar={"title": "Normalised to KPI max"},
            customdata=strategy_raw.values,
            hovertemplate=(
                "<b>%{y}</b><br>" + "%{x}<br>" + "Normalised score: %{z:.3f}<br>" + "Mean raw value: %{customdata:.4g}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        title=cfg["title"],
        template="plotly_white",
        width=950,
        height=max(360, 110 + 60 * len(strategy_y)),
        margin=dict(l=220, r=40, t=70, b=70),
    )
    fig.update_xaxes(tickangle=20)
    fig.update_yaxes(title_text=cfg["strategy_axis_title"])
    return fig, strategy_raw, strategy_norm

def build_delta_to_baseline_chart(config_key: str):
    cfg = STRATEGY_HEATMAP_CONFIG[config_key]
    try:
        cfg, plotted_delta_pct, plotted_raw, baseline = get_strategy_delta_data(config_key)
    except ValueError as exc:
        fig = _placeholder_figure(cfg["delta_title"], str(exc), width=1100, height=520)
        return fig, pd.DataFrame(), pd.Series(dtype=float)

    delta_colors = color_seq if "color_seq" in globals() else [
        "rgb(102,194,165)", "rgb(252,141,98)", "rgb(141,160,203)", "rgb(231,138,195)", "rgb(166,216,84)",
    ]
    fig = go.Figure()
    for idx, kpi in enumerate(plotted_delta_pct.columns):
        fig.add_trace(
            go.Bar(
                x=plotted_delta_pct.index,
                y=plotted_delta_pct[kpi],
                name=STRATEGY_DELTA_KPIS[kpi],
                marker_color=delta_colors[idx % len(delta_colors)],
                width=0.13,
                customdata=np.column_stack([
                    plotted_raw[kpi].values,
                    np.repeat(float(baseline[kpi]), len(plotted_delta_pct.index)),
                ]),
                hovertemplate=(
                    "<b>%{x}</b><br>" + f"{STRATEGY_DELTA_KPIS[kpi]} delta: " + "%{y:.2f}%<br>" +
                    "Strategy value: %{customdata[0]:.4g}<br>" + "Baseline value: %{customdata[1]:.4g}<extra></extra>"
                ),
            )
        )

    fig.add_hline(y=0.0, line_width=1.2, line_color="rgba(0,0,0,0.45)")
    fig.update_layout(
        title=cfg["delta_title"], template="plotly_white", width=1100, height=520,
        barmode="group", bargap=0.38, bargroupgap=0.14,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        margin=dict(l=90, r=40, t=90, b=90),
    )
    fig.update_xaxes(title_text=cfg["strategy_axis_title"], tickangle=20)
    fig.update_yaxes(title_text="Delta to baseline (%)", showgrid=True, gridcolor="rgba(0,0,0,0.10)", zeroline=False)
    return fig, plotted_delta_pct, baseline

def build_combined_delta_table(config_keys: list[str] | None = None) -> pd.DataFrame:
    keys = config_keys or STRATEGY_COMPARISON_ORDER
    frames = []
    for key in keys:
        cfg = STRATEGY_HEATMAP_CONFIG[key]
        try:
            _, plotted_delta_pct, _, _ = get_strategy_delta_data(key)
        except ValueError:
            continue
        long_df = plotted_delta_pct.reset_index(names="strategy").melt(id_vars="strategy", var_name="kpi", value_name="delta_pct")
        long_df["mode"] = cfg["mode_name"]
        long_df["config_key"] = key
        long_df["kpi_label"] = long_df["kpi"].map(KPI_LABELS_MAIN).fillna(long_df["kpi"])
        long_df["kpi_short"] = long_df["kpi"].map(STRATEGY_DELTA_KPIS)
        frames.append(long_df)

    if not frames:
        return pd.DataFrame(columns=["strategy", "kpi", "delta_pct", "mode", "config_key", "kpi_label", "kpi_short"])
    return pd.concat(frames, ignore_index=True)

def build_delta_heatmap(config_key: str, shared_abs_max: float | None = None):
    cfg = STRATEGY_HEATMAP_CONFIG[config_key]
    try:
        cfg, plotted_delta_pct, _, _ = get_strategy_delta_data(config_key)
    except ValueError as exc:
        fig = _placeholder_figure(cfg["delta_heatmap_title"], str(exc), width=900, height=360)
        return fig, pd.DataFrame(), shared_abs_max if shared_abs_max is not None else 5.0

    heat_tbl = plotted_delta_pct.reindex(columns=[k for k in STRATEGY_DELTA_KPIS if k in plotted_delta_pct.columns])
    heat_text = heat_tbl.map(lambda v: f"{v:+.0f}%").values
    if shared_abs_max is None:
        combined_delta = build_combined_delta_table()
        shared_abs_max = 5.0 if combined_delta.empty else float(np.nanmax(np.abs(combined_delta["delta_pct"].values)))
    shared_abs_max = max(5.0, float(shared_abs_max))

    fig = go.Figure(
        data=go.Heatmap(
            z=heat_tbl.values,
            x=[KPI_LABELS_MAIN.get(k, k) for k in heat_tbl.columns],
            y=heat_tbl.index.tolist(),
            text=heat_text,
            texttemplate="%{text}",
            textfont={"size": 10},
            colorscale="RdYlGn",
            zmid=0.0,
            zmin=-shared_abs_max,
            zmax=shared_abs_max,
            colorbar={"title": "Delta (%)"},
            hovertemplate="<b>%{y}</b><br>" + "%{x}<br>" + "Delta to baseline: %{z:.2f}%<extra></extra>",
        )
    )
    fig.update_layout(
        title=cfg["delta_heatmap_title"], template="plotly_white", width=900,
        height=max(320, 120 + 55 * len(heat_tbl.index)), margin=dict(l=210, r=40, t=85, b=80),
    )
    fig.update_xaxes(tickangle=25)
    fig.update_yaxes(title_text=cfg["strategy_axis_title"], autorange="reversed")
    return fig, heat_tbl, shared_abs_max

print("Strategy plot configuration ready for heatmaps and delta-to-baseline charts.")

Strategy plot configuration ready for heatmaps and delta-to-baseline charts.


In [27]:
# Delta-chart presentation override: group bars by KPI and colour them by strategy.
def build_delta_to_baseline_chart(config_key: str):
    cfg = STRATEGY_HEATMAP_CONFIG[config_key]
    try:
        cfg, plotted_delta_pct, plotted_raw, baseline = get_strategy_delta_data(config_key)
    except ValueError as exc:
        fig = _placeholder_figure(cfg["delta_title"], str(exc), width=1100, height=520)
        return fig, pd.DataFrame(), pd.Series(dtype=float)

    strategy_colors = color_seq if "color_seq" in globals() else [
        "rgb(102,194,165)", "rgb(252,141,98)", "rgb(141,160,203)", "rgb(231,138,195)", "rgb(166,216,84)",
    ]
    kpi_order = list(plotted_delta_pct.columns)
    kpi_labels = [STRATEGY_DELTA_KPIS[kpi] for kpi in kpi_order]
    fig = go.Figure()

    for idx, strategy in enumerate(plotted_delta_pct.index):
        fig.add_trace(
            go.Bar(
                x=kpi_labels,
                y=plotted_delta_pct.loc[strategy, kpi_order],
                name=str(strategy),
                marker_color=strategy_colors[idx % len(strategy_colors)],
                width=0.16,
                customdata=np.column_stack([
                    plotted_raw.loc[strategy, kpi_order].values,
                    baseline[kpi_order].values,
                ]),
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    + f"Strategy: {strategy}<br>"
                    + "Delta: %{y:.2f}%<br>"
                    + "Strategy value: %{customdata[0]:.4g}<br>"
                    + "Baseline value: %{customdata[1]:.4g}<extra></extra>"
                ),
            )
        )

    fig.add_hline(y=0.0, line_width=1.2, line_color="rgba(0,0,0,0.45)")
    fig.update_layout(
        title=cfg["delta_title"],
        template="plotly_white",
        width=1100,
        height=520,
        barmode="group",
        bargap=0.30,
        bargroupgap=0.12,
        legend_title_text=cfg["strategy_axis_title"].capitalize(),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        margin=dict(l=90, r=40, t=110, b=90),
    )
    fig.update_xaxes(title_text="KPI")
    fig.update_yaxes(title_text="Delta to constant baseline (%)", showgrid=True, gridcolor="rgba(0,0,0,0.10)", zeroline=False)
    return fig, plotted_delta_pct, baseline

print("Delta charts group bars by KPI and use colour to distinguish strategies.")

Delta charts group bars by KPI and use colour to distinguish strategies.


### 6.2 Strategy heatmaps

Each heatmap presents the mean KPI values for available strategies, normalised to the maximum of each KPI within that operating mode.

In [28]:
# Fed-batch feed strategy heatmap
fig_fb_heat, fb_heat_raw, fb_heat_norm = build_strategy_heatmap("fed_batch")
fig_fb_heat.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["fed_batch"]["output_name"]
    save_figure_svg(fig_fb_heat, out_svg, width=950, height=max(360, 110 + 60 * len(fb_heat_raw.index)))
    print(f"Saved: {out_svg.name}")

In [29]:
# Concentrated fed-batch perfusion strategy heatmap
fig_cf_heat, cf_heat_raw, cf_heat_norm = build_strategy_heatmap("concentrated_fedbatch")
fig_cf_heat.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["concentrated_fedbatch"]["output_name"]
    save_figure_svg(fig_cf_heat, out_svg, width=950, height=max(360, 110 + 60 * len(cf_heat_raw.index)))
    print(f"Saved: {out_svg.name}")

In [30]:
# Continuous perfusion strategy heatmap
fig_cp_heat, cp_heat_raw, cp_heat_norm = build_strategy_heatmap("continuous_perfusion")
fig_cp_heat.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["continuous_perfusion"]["output_name"]
    save_figure_svg(fig_cp_heat, out_svg, width=950, height=max(360, 110 + 60 * len(cp_heat_raw.index)))
    print(f"Saved: {out_svg.name}")

### 6.3 Delta to constant-strategy baselines

The individual charts and the combined three-panel chart report percentage changes relative to the configured constant feed or perfusion baseline.

In [31]:
# Fed-batch delta-to-baseline chart
fig_fb_delta, fb_delta_pct, fb_delta_baseline = build_delta_to_baseline_chart("fed_batch")
fig_fb_delta.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["fed_batch"]["delta_output_name"]
    save_figure_svg(fig_fb_delta, out_svg, width=1100, height=520)
    print(f"Saved: {out_svg.name}")

In [32]:
# Concentrated fed-batch delta-to-baseline chart
fig_cf_delta, cf_delta_pct, cf_delta_baseline = build_delta_to_baseline_chart("concentrated_fedbatch")
fig_cf_delta.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["concentrated_fedbatch"]["delta_output_name"]
    save_figure_svg(fig_cf_delta, out_svg, width=1100, height=520)
    print(f"Saved: {out_svg.name}")

In [33]:
# Continuous perfusion delta-to-baseline chart
fig_cp_delta, cp_delta_pct, cp_delta_baseline = build_delta_to_baseline_chart("continuous_perfusion")
fig_cp_delta.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["continuous_perfusion"]["delta_output_name"]
    save_figure_svg(fig_cp_delta, out_svg, width=1100, height=520)
    print(f"Saved: {out_svg.name}")

In [34]:
# One 1x3 grouped-bar figure with one panel per mode
combined_delta = build_combined_delta_table()
mode_panels = [
    STRATEGY_HEATMAP_CONFIG[key]["mode_name"]
    for key in STRATEGY_COMPARISON_ORDER
]
delta_colors = color_seq if "color_seq" in globals() else [
    "rgb(102,194,165)",
    "rgb(252,141,98)",
    "rgb(141,160,203)",
    "rgb(231,138,195)",
    "rgb(166,216,84)",
]
kpi_panel_order = [k for k in STRATEGY_DELTA_KPIS if k in combined_delta["kpi"].unique()]
y_abs_max = float(np.nanmax(np.abs(combined_delta["delta_pct"].values)))
y_abs_max = max(5.0, y_abs_max)
y_range = [-1.1 * y_abs_max, 1.1 * y_abs_max]

fig_delta_panels = make_subplots(
    rows=1,
    cols=len(mode_panels),
    subplot_titles=mode_panels,
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

for col_idx, mode_name in enumerate(mode_panels, start=1):
    sub = combined_delta.loc[combined_delta["mode"].eq(mode_name)].copy()
    strategy_order = list(dict.fromkeys(sub["strategy"].tolist()))

    for kpi_idx, kpi in enumerate(kpi_panel_order):
        trace_df = sub.loc[sub["kpi"].eq(kpi)].copy()
        trace_df["strategy"] = pd.Categorical(trace_df["strategy"], categories=strategy_order, ordered=True)
        trace_df = trace_df.sort_values("strategy")

        fig_delta_panels.add_trace(
            go.Bar(
                x=trace_df["strategy"],
                y=trace_df["delta_pct"],
                name=STRATEGY_DELTA_KPIS[kpi],
                marker_color=delta_colors[kpi_idx % len(delta_colors)],
                width=0.13,
                showlegend=(col_idx == 1),
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    + f"{STRATEGY_DELTA_KPIS[kpi]} delta: "
                    + "%{y:.2f}%<extra></extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )

    fig_delta_panels.add_hline(y=0.0, line_width=1.2, line_color="rgba(0,0,0,0.45)", row=1, col=col_idx)
    fig_delta_panels.update_xaxes(tickangle=25, row=1, col=col_idx)
    fig_delta_panels.update_yaxes(range=y_range, row=1, col=col_idx)

fig_delta_panels.update_layout(
    title="Delta to Constant Strategy Across Modes",
    template="plotly_white",
    width=1500,
    height=520,
    barmode="group",
    bargap=0.34,
    bargroupgap=0.12,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=70, r=30, t=90, b=90),
)
fig_delta_panels.update_yaxes(title_text="Delta to baseline (%)", showgrid=True, gridcolor="rgba(0,0,0,0.10)")

fig_delta_panels.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / f"{CAMPAIGN_LABEL.lower()}_combined_delta_bars_3panel.svg"
    save_figure_svg(fig_delta_panels, out_svg, width=1500, height=520)
    print(f"Saved: {out_svg.name}")

### 6.4 Delta heatmaps with a shared colour scale

All three heatmaps use the same absolute percentage scale to make strategy effects comparable across operating modes.

In [35]:
# Delta heatmap: fed-batch
shared_delta_abs_max = max(
    5.0,
    float(np.nanmax(np.abs(build_combined_delta_table()["delta_pct"].values)))
    if not build_combined_delta_table().empty else 5.0,
 )

fig_fb_delta_heatmap, fb_delta_heat_tbl, shared_delta_abs_max = build_delta_heatmap(
    "fed_batch",
    shared_abs_max=shared_delta_abs_max,
 )
fig_fb_delta_heatmap.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["fed_batch"]["delta_heatmap_output_name"]
    save_figure_svg(
        fig_fb_delta_heatmap,
        out_svg,
        width=900,
        height=max(320, 120 + 55 * len(fb_delta_heat_tbl.index)),
    )
    print(f"Saved: {out_svg.name}")

In [36]:
# Delta heatmap: concentrated fed-batch
if "shared_delta_abs_max" not in globals():
    shared_delta_abs_max = max(
        5.0,
        float(np.nanmax(np.abs(build_combined_delta_table()["delta_pct"].values)))
        if not build_combined_delta_table().empty else 5.0,
    )

fig_cf_delta_heatmap, cf_delta_heat_tbl, shared_delta_abs_max = build_delta_heatmap(
    "concentrated_fedbatch",
    shared_abs_max=shared_delta_abs_max,
 )
fig_cf_delta_heatmap.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["concentrated_fedbatch"]["delta_heatmap_output_name"]
    save_figure_svg(
        fig_cf_delta_heatmap,
        out_svg,
        width=900,
        height=max(320, 120 + 55 * len(cf_delta_heat_tbl.index)),
    )
    print(f"Saved: {out_svg.name}")

In [37]:
# Delta heatmap: continuous perfusion
if "shared_delta_abs_max" not in globals():
    shared_delta_abs_max = max(
        5.0,
        float(np.nanmax(np.abs(build_combined_delta_table()["delta_pct"].values)))
        if not build_combined_delta_table().empty else 5.0,
    )

fig_cp_delta_heatmap, cp_delta_heat_tbl, shared_delta_abs_max = build_delta_heatmap(
    "continuous_perfusion",
    shared_abs_max=shared_delta_abs_max,
 )
fig_cp_delta_heatmap.show()

SAVE_FIG = False
if SAVE_FIG:
    out_svg = NOTEBOOK_DIR / STRATEGY_HEATMAP_CONFIG["continuous_perfusion"]["delta_heatmap_output_name"]
    save_figure_svg(
        fig_cp_delta_heatmap,
        out_svg,
        width=900,
        height=max(320, 120 + 55 * len(cp_delta_heat_tbl.index)),
    )
    print(f"Saved: {out_svg.name}")